- **Fundamental Analysis**
    - Pick a public stock of your choosing
    - Compute the following ratios: metrics.csv

    - You may use excel or its equivalents, or python to do so. Use the company’s financial reports for the numbers.
    - Also write a small analysis (100 words) on the financial health of the company.
- **Model Design**
    - From the list of ratios on the Fundamental Analysis Basics page, filter a list of parameters you use for your model.
    - Provide justifications for the parameters chosen.
    - If you feel like there is a ratio or variable that would be useful but is not present, you may suggest it to me and I will obtain the data.

### Metrices Calculations for Fundamental Analysis
Ticker of the choosen company : ETERNAL

In [10]:
# Financial Data from Annual Report FY25
financials = {
    # Income Statement
    "revenue": 20243,
    "ebitda": 637,
    "pat": 527,
    "gross_profit": 20243 - 5565,  # Revenue - COGS
    "interest_expense": 154,
    "depreciation": 863,

    # Balance Sheet
    "total_assets": 65000,
    "total_equity": 52000,
    "total_debt": 1800,
    "capital_employed": 53800,  # Equity + Debt - Cash
    "fixed_assets": 7200,
    "working_capital": 4800,
    "inventory": 2100,
    "receivables": 3600,

    # Share Data
    "shares_outstanding": 94.5,   # crore shares
    "market_price": 245,          # ₹ per share

    # Previous Year Data (for Growth/CAGR)
    "revenue_prev": 18500,
    "ebitda_prev": 580,
    "pat_prev": 450,
    "eps_prev": 450 / 94.5
}


In [11]:
#profitability ratios
def ebitda_margin(data):
    return data["ebitda"] / data["revenue"] * 100

def pat_margin(data):
    return data["pat"] / data["revenue"] * 100

def gross_profit_margin(data):
    return data["gross_profit"] / data["revenue"] * 100

def roe(data):
    return data["pat"] / data["total_equity"] * 100

def roa(data):
    return data["pat"] / data["total_assets"] * 100

def roce(data):
    return data["ebitda"] / data["capital_employed"] * 100
def growth_rate(current, previous):
    return (current - previous) / previous * 100

def cagr(start, end, years):
    return ((end / start) ** (1 / years) - 1) * 100



In [12]:
#Leverage/Solvency Ratios
def debt_equity_ratio(data):
    return data["total_debt"] / data["total_equity"]

def leverage_ratio(data):
    return data["total_assets"] / data["total_equity"]

def interest_coverage(data):
    return data["ebitda"] / data["interest_expense"]


In [13]:
#Efficiency Ratios
def fixed_asset_turnover(data):
    return data["revenue"] / data["fixed_assets"]

def working_capital_turnover(data):
    return data["revenue"] / data["working_capital"]

def total_asset_turnover(data):
    return data["revenue"] / data["total_assets"]

def inventory_turnover(data):
    return data["revenue"] / data["inventory"]

def inventory_days(data):
    return 365 / inventory_turnover(data)

def receivables_turnover(data):
    return data["revenue"] / data["receivables"]

def dso(data):
    return 365 / receivables_turnover(data)


In [14]:
#Valuation Ratios
def eps(data):
    return data["pat"] / data["shares_outstanding"]

def book_value_per_share(data):
    return data["total_equity"] / data["shares_outstanding"]

def pe_ratio(data):
    return data["market_price"] / eps(data)

def pb_ratio(data):
    return data["market_price"] / book_value_per_share(data)

def ps_ratio(data):
    return data["market_price"] / (data["revenue"] / data["shares_outstanding"])


In [15]:
#Values
fcf = 800                  # ₹ crore
growth = 0.10              # 10%
discount = 0.12            # 12%
terminal_growth = 0.04     # 4%
shares = 94.5              # crore shares
market_price = 245         # ₹

In [16]:
#Valuation Models
def dcf(fcf, growth, discount, years=10):
    value = 0
    for t in range(1, years + 1):
        value += (fcf * (1 + growth)**t) / (1 + discount)**t
    return value
dcf_value = dcf(fcf, growth, discount)
def intrinsic_value(dcf_value, shares):
    return dcf_value / shares
intrinsic = intrinsic_value(dcf_value, shares)
def margin_of_safety(intrinsic, market):
    return (intrinsic - market) / intrinsic * 100

def fair_value_band(intrinsic):
    return (intrinsic * 0.9, intrinsic * 1.1)


In [17]:

def calculate_all_ratios(data):
    ebitda_margin_curr = ebitda_margin(data)
    ebitda_margin_prev = data['ebitda_prev'] / data['revenue_prev'] * 100
    pat_margin_curr = pat_margin(data)
    pat_margin_prev = data['pat_prev'] / data['revenue_prev'] * 100
    
    ratios = [
        {'Category': 'Profitability', 'Metric': 'EBITDA Margin', 'Value': ebitda_margin_curr},
        {'Category': 'Profitability', 'Metric': 'PAT Margin (Net Profit Margin)', 'Value': pat_margin_curr},
        {'Category': 'Profitability', 'Metric': 'EBITDA Margin CAGR', 'Value': cagr(ebitda_margin_prev, ebitda_margin_curr, 1)},
        {'Category': 'Profitability', 'Metric': 'PAT Margin CAGR', 'Value': cagr(pat_margin_prev, pat_margin_curr, 1)},
        {'Category': 'Profitability', 'Metric': 'Return on Equity (ROE)', 'Value': roe(data)},
        {'Category': 'Profitability', 'Metric': 'Return on Assets (ROA)', 'Value': roa(data)},
        {'Category': 'Profitability', 'Metric': 'Return on Capital Employed (ROCE)', 'Value': roce(data)},
        {'Category': 'Profitability', 'Metric': 'Gross Profit Margin (GPM)', 'Value': gross_profit_margin(data)},
        {'Category': 'Profitability', 'Metric': 'Net Profit Growth', 'Value': growth_rate(data['pat'], data['pat_prev'])},
        {'Category': 'Profitability', 'Metric': 'EPS Growth / Trend', 'Value': growth_rate(eps(data), data['eps_prev'])},
        
        {'Category': 'Leverage/Solvency', 'Metric': 'Leverage Ratio (Debt vs Equity)', 'Value': leverage_ratio(data)},
        {'Category': 'Leverage/Solvency', 'Metric': 'Debt/Equity Ratio', 'Value': debt_equity_ratio(data)},
        {'Category': 'Leverage/Solvency', 'Metric': 'Interest Coverage Ratio', 'Value': interest_coverage(data)},
        
        {'Category': 'Efficiency', 'Metric': 'Fixed Asset Turnover Ratio', 'Value': fixed_asset_turnover(data)},
        {'Category': 'Efficiency', 'Metric': 'Working Capital Turnover Ratio', 'Value': working_capital_turnover(data)},
        {'Category': 'Efficiency', 'Metric': 'Total Assets Turnover Ratio', 'Value': total_asset_turnover(data)},
        {'Category': 'Efficiency', 'Metric': 'Inventory Turnover Ratio', 'Value': inventory_turnover(data)},
        {'Category': 'Efficiency', 'Metric': 'Inventory Days', 'Value': inventory_days(data)},
        {'Category': 'Efficiency', 'Metric': 'Receivables Turnover Ratio', 'Value': receivables_turnover(data)},
        {'Category': 'Efficiency', 'Metric': 'Days Sales Outstanding (DSO)', 'Value': dso(data)},
        
        {'Category': 'Valuation', 'Metric': 'Price to Earnings (P/E) Ratio', 'Value': pe_ratio(data)},
        {'Category': 'Valuation', 'Metric': 'Price to Book (P/B) Ratio', 'Value': pb_ratio(data)},
        {'Category': 'Valuation', 'Metric': 'Price to Sales (P/S) Ratio', 'Value': ps_ratio(data)},
        {'Category': 'Valuation', 'Metric': 'Earnings Per Share (EPS)', 'Value': eps(data)},
        {'Category': 'Valuation', 'Metric': 'Book Value per Share', 'Value': book_value_per_share(data)},
        
        {'Category': 'Valuation Model', 'Metric': 'Discounted Cash Flow (DCF)', 'Value': dcf(fcf, growth, discount)},
        {'Category': 'Valuation Model', 'Metric': 'Intrinsic Value per Share', 'Value': intrinsic},
        {'Category': 'Valuation Model', 'Metric': 'Fair Value Band', 'Value': str(fair_value_band(intrinsic))},
        {'Category': 'Valuation Model', 'Metric': 'Margin of Safety', 'Value': margin_of_safety(intrinsic, market_price)}
    ]
    return ratios

ratios_data = calculate_all_ratios(financials)


In [19]:
import pandas as pd

df = pd.DataFrame(ratios_data)
df.to_csv("Financial_metrics_ETERNAL_FY25.csv", index=False)
df.head()


,Category,Metric,Value
0,Profitability,EBITDA Margin,3.146767
1,Profitability,PAT Margin (Net Profit Margin),2.603369
2,Profitability,EBITDA Margin CAGR,0.371009
3,Profitability,PAT Margin CAGR,7.027395
4,Profitability,Return on Equity (ROE),1.013462


In [2]:
# Filtered metrics for model design

import pandas as pd

selected_metrics = [
    {
        "Category": "Profitability",
        "Metric": "EBITDA Margin",
        "Justification": "Measures operating efficiency before financing and accounting effects."
    },
    {
        "Category": "Profitability",
        "Metric": "PAT Margin (Net Profit Margin)",
        "Justification": "Shows the percentage of revenue converted into net profit."
    },
    {
        "Category": "Profitability",
        "Metric": "Return on Equity (ROE)",
        "Justification": "Measures how efficiently shareholder capital generates profits."
    },
    {
        "Category": "Profitability",
        "Metric": "Return on Capital Employed (ROCE)",
        "Justification": "Evaluates how effectively the company utilizes its capital."
    },
    {
        "Category": "Profitability",
        "Metric": "Return on Assets (ROA)",
        "Justification": "Indicates how efficiently assets generate profits."
    },
    {
        "Category": "Growth",
        "Metric": "Net Profit Growth",
        "Justification": "Captures the company's earnings growth over time."
    },
    {
        "Category": "Growth",
        "Metric": "EPS Growth / Trend",
        "Justification": "Reflects the growth in earnings available to shareholders."
    },
    {
        "Category": "Growth",
        "Metric": "EBITDA Margin CAGR",
        "Justification": "Measures long-term improvement in operating profitability."
    },
    {
        "Category": "Growth",
        "Metric": "PAT Margin CAGR",
        "Justification": "Indicates sustained improvement in net profitability."
    },
    {
        "Category": "Leverage",
        "Metric": "Debt-to-Equity Ratio",
        "Justification": "Measures financial leverage and capital structure risk."
    },
    {
        "Category": "Leverage",
        "Metric": "Interest Coverage Ratio",
        "Justification": "Assesses the company's ability to meet interest obligations."
    },
    {
        "Category": "Efficiency",
        "Metric": "Total Asset Turnover Ratio",
        "Justification": "Measures how efficiently assets generate revenue."
    },
    {
        "Category": "Efficiency",
        "Metric": "Working Capital Turnover Ratio",
        "Justification": "Evaluates efficiency in utilizing working capital."
    },
    {
        "Category": "Valuation",
        "Metric": "Price-to-Earnings (P/E) Ratio",
        "Justification": "Represents the market valuation relative to earnings."
    },
    {
        "Category": "Valuation",
        "Metric": "Price-to-Book (P/B) Ratio",
        "Justification": "Measures market value relative to the company's book value."
    },
    {
        "Category": "Valuation",
        "Metric": "Discounted Cash Flow (DCF)",
        "Justification": "Estimates the intrinsic value based on future cash flows."
    },
    {
        "Category": "Valuation",
        "Metric": "Margin of Safety",
        "Justification": "Indicates the discount of market price from intrinsic value."
    },
    {
        "Category": "Liquidity",
        "Metric": "Current Ratio",
        "Justification": "Measures the company's ability to meet short-term obligations."
    }
]

# Create DataFrame
filtered_metrics4model = pd.DataFrame(selected_metrics)

# Save to CSV
filtered_metrics4model.to_csv("filtered_metrics4model.csv", index=False)

print("File saved as 'filtered_metrics4model.csv'")
display(filtered_metrics4model)

File saved as 'filtered_metrics4model.csv'


,Category,Metric,Justification
0,Profitability,EBITDA Margin,Measures operating efficiency before financing...
1,Profitability,PAT Margin (Net Profit Margin),Shows the percentage of revenue converted into...
2,Profitability,Return on Equity (ROE),Measures how efficiently shareholder capital g...
3,Profitability,Return on Capital Employed (ROCE),Evaluates how effectively the company utilizes...
4,Profitability,Return on Assets (ROA),Indicates how efficiently assets generate prof...
5,Growth,Net Profit Growth,Captures the company's earnings growth over time.
6,Growth,EPS Growth / Trend,Reflects the growth in earnings available to s...
7,Growth,EBITDA Margin CAGR,Measures long-term improvement in operating pr...
8,Growth,PAT Margin CAGR,Indicates sustained improvement in net profita...
9,Leverage,Debt-to-Equity Ratio,Measures financial leverage and capital struct...


###  Small analysis (100 words) on the financial health of the company.
The FY25 financial metrics show that the company is in a strong financial position. It has good profit growth, with both net profit and earnings per share increasing by around 17%. The company also has very low debt, which reduces financial risk, and it earns enough profit to comfortably pay its interest expenses. Its asset and working capital turnover ratios show that resources are being used efficiently. However, the profit margins are still relatively low, which is common in this industry. The negative margin of safety suggests that the company's stock price is higher than its estimated value, so it may currently be overvalued in the market.
